# VPA Demo: YAML Workflow

This notebook writes a small workflow under `/tmp`, runs it with CPU-safe tools,
inspects the workflow record, and generates a report. It mirrors
`docs/use_workflows.md`.

In [ ]:
from __future__ import annotations

import json
import os
import shlex
import shutil
import subprocess
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for path in [here, *here.parents]:
        if (path / "pyproject.toml").is_file() and (path / "src" / "vibeproteinagent").is_dir():
            return path
    return here


REPO_ROOT = find_repo_root()
RUN_INSTALL = os.environ.get("VPA_DEMO_RUN_INSTALL", "0") == "1"
RUN_HEAVY = os.environ.get("VPA_DEMO_RUN_HEAVY", "0") == "1"
RUN_LLM = os.environ.get("VPA_DEMO_RUN_LLM", "0") == "1"
WORKSPACE_PREFIX = os.environ.get("VPA_DEMO_WORKSPACE_PREFIX", f"vpa_demo_{os.getpid()}")
OUT_DIR = Path(os.environ.get("VPA_DEMO_OUT_DIR", "/tmp")) / WORKSPACE_PREFIX
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep the demo daemon/socket isolated from any VPA daemon the user may already be running.
os.environ.setdefault("VPA_RUNTIME_DIR", str(OUT_DIR / "runtime"))
Path(os.environ["VPA_RUNTIME_DIR"]).mkdir(parents=True, exist_ok=True)


def _candidate_vpa_commands() -> list[list[str]]:
    commands: list[list[str]] = []
    configured = os.environ.get("VPA_COMMAND")
    if configured:
        commands.append(shlex.split(configured))

    repo_venv_vpa = REPO_ROOT / ".venv" / "bin" / "vpa"
    if repo_venv_vpa.is_file():
        commands.append([str(repo_venv_vpa)])

    installed = shutil.which("vpa")
    if installed:
        commands.append([installed])

    if (REPO_ROOT / "pyproject.toml").is_file() and shutil.which("uv"):
        commands.append(["uv", "run", "vpa"])

    deduped: list[list[str]] = []
    seen: set[tuple[str, ...]] = set()
    for cmd in commands:
        key = tuple(cmd)
        if key not in seen:
            deduped.append(cmd)
            seen.add(key)
    return deduped


def choose_vpa() -> list[str]:
    errors = []
    for base in _candidate_vpa_commands():
        proc = subprocess.run(
            [*base, "version", "--json"],
            cwd=REPO_ROOT,
            text=True,
            capture_output=True,
            env=os.environ.copy(),
            timeout=120,
        )
        if proc.returncode == 0:
            print("Using VPA command:", " ".join(shlex.quote(part) for part in base))
            print(proc.stdout.strip())
            return base
        errors.append((base, proc.returncode, proc.stderr.strip() or proc.stdout.strip()))
    raise RuntimeError(f"Could not find a working vpa command: {errors}")


VPA = choose_vpa()


def show_command(parts: list[str]) -> None:
    print("$ " + " ".join(shlex.quote(str(part)) for part in parts))


def run(parts: list[str], *, check: bool = True, timeout: int = 300, cwd: Path | None = None) -> subprocess.CompletedProcess[str]:
    parts = [str(part) for part in parts]
    show_command(parts)
    proc = subprocess.run(
        parts,
        cwd=cwd or REPO_ROOT,
        text=True,
        capture_output=True,
        env=os.environ.copy(),
        timeout=timeout,
    )
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if check and proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, parts, proc.stdout, proc.stderr)
    return proc


def run_json(parts: list[str], *, timeout: int = 300, check: bool = True) -> dict:
    proc = run([*parts, "--json"], timeout=timeout, check=check)
    if proc.returncode != 0:
        return {}
    return json.loads(proc.stdout)


def stop_demo_daemon() -> None:
    run([*VPA, "daemon", "stop", "--workspace", f"{WORKSPACE_PREFIX}_cleanup", "--json"], check=False, timeout=60)


print("Repo root:", REPO_ROOT)
print("Output directory:", OUT_DIR)
print("Runtime directory:", os.environ["VPA_RUNTIME_DIR"])
print("RUN_INSTALL=", RUN_INSTALL, "RUN_HEAVY=", RUN_HEAVY, "RUN_LLM=", RUN_LLM)


## Optional Install

In [ ]:
install_cmd = ["npx", "vibeproteinagent", "install", "--profile", "base", "--cuda-profile", "auto", "--yes"]
if RUN_INSTALL:
    run(install_cmd, timeout=3600)
else:
    print("Skipping install. To run it, set VPA_DEMO_RUN_INSTALL=1.")
    show_command(install_cmd)

## 1. Write The Workflow YAML

The workflow mutates a sequence, analyzes the mutant, runs QC, then ranks the
candidate emitted by QC. The run parameters are written separately so the demo
also verifies `--params-file`.


In [ ]:
workflow_path = OUT_DIR / "sequence_triage.yaml"
workflow_yaml = """
name: sequence_triage
version: "1.0.0"
description: "Mutate a sequence, analyze the mutant, run QC, and rank it."
params:
  sequence:
    type: str
    required: true
  mapping:
    type: dict
    default: {"3": "A"}
nodes:
  - id: mutate
    tool: sequence_edit
    inputs:
      operation: mutate_residues
      sequence: ${params.sequence}
      mapping: ${params.mapping}
      output_name: demo_mutant
    cache: true
  - id: analyze
    tool: sequence_analysis
    inputs:
      sequence: ${outputs.mutate.mutant_sequence}
    cache: true
  - id: qc
    tool: protein_qc
    inputs:
      sequence: ${outputs.mutate.mutant_sequence}
      metrics: ["gravy", "instability_index", "pi", "polybasic"]
    cache: true
  - id: rank
    tool: rank_designs
    inputs:
      candidates: ${outputs.qc.candidates}
      qc_results: ${outputs.qc}
      top_k: 1
      output_prefix: sequence_triage_ranked
    cache: true
""".lstrip()
workflow_path.write_text(workflow_yaml, encoding="utf-8")
params_path = OUT_DIR / "workflow_params.json"
params_path.write_text(
    json.dumps({"sequence": "ACDEFGHIKLMNPQRSTVWY", "mapping": {"3": "A", "9": "Q"}}, indent=2),
    encoding="utf-8",
)
print(workflow_path)
print(workflow_path.read_text(encoding="utf-8"))
print(params_path)
print(params_path.read_text(encoding="utf-8"))


## 2. Inspect Tools And Workflow

In [ ]:
for tool in ["sequence_edit", "sequence_analysis", "protein_qc", "rank_designs"]:
    run_json([*VPA, "tool", "info", tool], timeout=120)
run_json([*VPA, "workflow", "inspect", str(workflow_path)], timeout=120)

## 3. Start A CPU Worker, Prewarm Tools, And Run The Workflow

The workflow runs synchronously once a daemon is available. Parameters are read
from the JSON file written above. Prewarm verifies that the workflow adapters can
load before the workflow is submitted.


In [ ]:
workspace = f"{WORKSPACE_PREFIX}_workflow"
stop_demo_daemon()
run_json([*VPA, "daemon", "start", "--cpu-workers", "1", "--workspace", workspace], timeout=120)
for tool in ["sequence_edit", "sequence_analysis", "protein_qc", "rank_designs"]:
    run_json([*VPA, "daemon", "warm", tool], timeout=120)
workflow_run = run_json([
    *VPA,
    "workflow",
    "run",
    str(workflow_path),
    "--workspace",
    workspace,
    "--params-file",
    str(params_path),
    "--max-wait-s",
    "3600",
], timeout=240)
workflow_id = workflow_run["data"]["workflow_id"]
print("Workflow id:", workflow_id)


## 4. Inspect, Resume Command, And Report

In [ ]:
status = run_json([*VPA, "workflow", "status", workflow_id, "--workspace", workspace], timeout=120)
print("Workflow state:", status["data"]["state"])
print("Resume command, if needed:")
show_command([*VPA, "workflow", "run", str(workflow_path), "--workspace", workspace, "--resume", workflow_id])
report_path = OUT_DIR / "workflow_report.html"
report = run_json([*VPA, "report", workflow_id, "--workspace", workspace, "--out", str(report_path)], timeout=120)
print("Report:", report["data"]["report_html"])

## 5. Optional Heavy Workflow Pattern

In [ ]:
heavy_cmds = [
    [*VPA, "install", "doctor", "--tools", "rfdiffusion,proteinmpnn,protenix_v2,protein_qc,rank_designs"],
    [*VPA, "daemon", "start", "--workers", "1", "--cpu-workers", "2", "--workspace", f"{WORKSPACE_PREFIX}_heavy_workflow"],
]
print("A heavyweight workflow usually follows: generator -> inverse folding -> prediction -> QC/analysis -> rank -> report.")
if RUN_HEAVY:
    for cmd in heavy_cmds:
        run(cmd, timeout=7200)
else:
    print("Skipping heavy workflow setup. To execute heavy examples, set VPA_DEMO_RUN_HEAVY=1.")
    for cmd in heavy_cmds:
        show_command(cmd)

## 6. Stop The Demo Daemon

In [ ]:
run([*VPA, "daemon", "stop", "--workspace", workspace, "--json"], check=False, timeout=60)